# 05 — Error Analysis Figures (Ch 4.10)

**Updated 2026-05-31 after WS1 completion:**
- `csqe_vs_blind_per_query.csv` now committed in `data/raw/` (was the hard blocker). Column names: `ndcg10_csqe_hybrid`, `ndcg10_aya_blind_bm25`, `delta_ndcg10_csqe_vs_blind`.
- Fig 4.14 uses Scheme A (1–3 / 4–8 / 9+ words) via `data/computed/sec4_10_length_buckets_1-3-4-8-9.csv`.
- Fig 4.13 uses validated first-pass definition: BM25 top-1 with qrel≥1.

**Outputs:**
- Fig 4.12 v1/v2 — per-query Δ NDCG@10 (hist/KDE)
- Fig 4.13 v1/v2 — NDCG@10 by 1st-pass relevance
- Fig 4.14 v1/v2 — Δ by query length (Scheme A)
- Fig 4.15 v1/v2 — regression types (stacked bar / donut)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _helpers import *

## Fig 4.12 — Per-query Δ histogram (CSQE+Hybrid vs Aya-blind BM25)

In [ ]:
delta_path = DATA_RAW / 'csqe_vs_blind_per_query.csv'
d = pd.read_csv(delta_path)
delta_col = 'delta_ndcg10_csqe_vs_blind'
print(f'rows: {len(d)} · mean delta: {d[delta_col].mean():+.4f} · CSQE>blind: {(d[delta_col]>0).mean()*100:.1f}%')

# v1 histogram with shaded improvement/regression bands
fig, ax = plt.subplots()
ax.hist(d[delta_col], bins=np.linspace(-1, 1, 41), color='#4d4d4d', edgecolor='white')
ax.axvline(0, color='#1f6f8a', linewidth=1)
ax.axvspan(-1, -0.1, alpha=0.10, color='#1f6f8a')
ax.axvspan(0.3, 1, alpha=0.10, color='#1f6f8a')
n_improve = (d[delta_col] > 0).sum()
n_reg = (d[delta_col] < -0.1).sum()
n_total = len(d)
ax.text(0.6, ax.get_ylim()[1] * 0.85,
        f'Improves: {n_improve}/{n_total} ({n_improve/n_total*100:.1f}%)',
        fontsize=8, ha='left')
ax.text(-0.95, ax.get_ylim()[1] * 0.85,
        f'Regresses (\u0394<-0.1):\n{n_reg}/{n_total} ({n_reg/n_total*100:.1f}%)',
        fontsize=8, ha='left')
ax.set_xlabel('Per-query \u0394 NDCG@10 (CSQE+Hybrid \u2212 Aya-blind BM25)')
ax.set_ylabel('Number of queries')
save_fig(fig, 'fig_4_12_delta_hist_v1')

# v2 KDE
from scipy import stats as sps
fig, ax = plt.subplots()
xs = np.linspace(-1, 1, 200)
kde = sps.gaussian_kde(d[delta_col])
ax.fill_between(xs, kde(xs), alpha=0.5, color='#4d4d4d')
ax.axvline(d[delta_col].mean(), color='#1f6f8a', linestyle='--', linewidth=1,
           label=f'Mean = {d[delta_col].mean():+.3f}')
ax.axvline(0, color='#8c8c8c', linewidth=0.7)
ax.set_xlabel('Per-query \u0394 NDCG@10 (CSQE+Hybrid \u2212 Aya-blind BM25)')
ax.set_ylabel('Density')
ax.legend(loc='upper left')
save_fig(fig, 'fig_4_12_delta_kde_v2')

## Fig 4.13 — NDCG@10 by 1st-pass relevance

First-pass relevance = BM25 baseline retrieves a relevant doc at rank 1 (qrel≥1). Counts confirmed in WS1: 1,061 relevant / 1,835 not-relevant. CSQE+Hybrid scores 0.8877 vs 0.5814.

In [ ]:
ep = pd.read_csv(DATA_RAW / 'csqe_error_patterns.csv')
subset = ep[ep.Category.isin(['1st-pass doc IS relevant', '1st-pass doc NOT relevant'])].copy()
subset['short'] = ['1st-pass relevant\n(BM25 top-1, qrel\u22651)', '1st-pass NOT relevant']
labels = subset.short.tolist()
x = np.arange(len(labels))
w = 0.35

# v1 plain
fig, ax = plt.subplots()
ax.bar(x - w/2, subset['Aya blind BM25 nDCG@10'], w, label='Aya-blind BM25', color='#8c8c8c', edgecolor='black')
ax.bar(x + w/2, subset['CSQE+Hybrid nDCG@10'], w, label='CSQE + Hybrid (BM25-only-expanded RRF)', color='#1f6f8a', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('NDCG@10')
ax.legend(loc='upper right', fontsize=8)
save_fig(fig, 'fig_4_13_firstpass_v1')

# v2 with sample-size annotations
fig, ax = plt.subplots()
b1 = ax.bar(x - w/2, subset['Aya blind BM25 nDCG@10'], w, label='Aya-blind BM25', color='#8c8c8c', edgecolor='black')
b2 = ax.bar(x + w/2, subset['CSQE+Hybrid nDCG@10'], w, label='CSQE + Hybrid', color='#1f6f8a', edgecolor='black')
for bar, v in zip(b1, subset['Aya blind BM25 nDCG@10']):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}',
            ha='center', va='bottom', fontsize=8)
for bar, v in zip(b2, subset['CSQE+Hybrid nDCG@10']):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}',
            ha='center', va='bottom', fontsize=8)
for i, n in enumerate(subset.n.tolist()):
    ax.text(x[i], -0.04, f'n = {n}', ha='center', va='top', fontsize=8,
            color='#4d4d4d', transform=ax.get_xaxis_transform())
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('NDCG@10')
ax.legend(loc='upper right', fontsize=8)
save_fig(fig, 'fig_4_13_firstpass_v2_annot')

## Fig 4.14 — Δ NDCG@10 by query length (Scheme A: 1–3 / 4–8 / 9+ words)

Source: `data/computed/sec4_10_length_buckets_1-3-4-8-9.csv` — produced by Mohammed in WS1 from the per-query CSV, using the unified Scheme A bucketing.

In [ ]:
buckets = pd.read_csv(DATA_COMPUTED / 'sec4_10_length_buckets_1-3-4-8-9.csv')
buckets = buckets.rename(columns={'b': 'bucket'})
print(buckets)
labels = buckets.bucket.tolist()

# v1 — Δ bars with relative gain annotation
fig, ax = plt.subplots()
ax.bar(labels, buckets.delta, color='#1f6f8a', edgecolor='black')
for i, row in buckets.iterrows():
    rel = row.delta / row.aya_blind * 100
    ax.text(i, row.delta + 0.005, f'+{row.delta:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i, row.delta * 0.5, f'(+{rel:.1f}% rel.)',
            ha='center', va='center', fontsize=8, color='white')
ax.set_ylabel('\u0394 NDCG@10 (CSQE+Hybrid \u2212 Aya-blind)')
ax.set_xlabel('Query length (words)')
save_fig(fig, 'fig_4_14_lengthgain_v1')

# v2 — grouped blind vs CSQE per bucket
x = np.arange(len(buckets))
w = 0.35
fig, ax = plt.subplots()
b1 = ax.bar(x - w/2, buckets.aya_blind, w, label='Aya-blind BM25', color='#8c8c8c', edgecolor='black')
b2 = ax.bar(x + w/2, buckets.csqe_hybrid, w, label='CSQE + Hybrid', color='#1f6f8a', edgecolor='black')
for bar, v in zip(b1, buckets.aya_blind):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
for bar, v in zip(b2, buckets.csqe_hybrid):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
for i, n in enumerate(buckets.n.tolist()):
    ax.text(x[i], -0.04, f'n = {n}', ha='center', va='top', fontsize=8,
            color='#4d4d4d', transform=ax.get_xaxis_transform())
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('NDCG@10')
ax.set_xlabel('Query length (words)')
ax.legend(loc='upper left', fontsize=8)
save_fig(fig, 'fig_4_14_lengthgain_v2_grouped')

## Fig 4.15 — Regression types (Type A: 52% / Type B: 36% / Type C: 12%)

Source: aggregate breakdown from `exp_error_analysis_csqe.md`. n=367 regressions (Δ<-0.1).

In [ ]:
regs = pd.DataFrame({
    'type': ['Type A\n(strong BM25 hurt)', 'Type B\n(poisoned 1st-pass)', 'Type C\n(other)'],
    'pct':  [52, 36, 12],
})

# v1 stacked horizontal bar
fig, ax = plt.subplots(figsize=(7, 1.8))
left = 0
colors = ['#1f6f8a', '#6d4a8f', '#8c8c8c']
for (_, r), c in zip(regs.iterrows(), colors):
    ax.barh([''], [r.pct], left=left, color=c, edgecolor='black', label=r.type)
    text_color = 'white' if c < '7' else 'black'
    ax.text(left + r.pct/2, 0, f'{r.type}\n{r.pct}%', ha='center', va='center',
            color=text_color, fontsize=8)
    left += r.pct
ax.set_xlim(0, 100)
ax.set_xlabel('% of regressions (n = 367)')
ax.set_yticks([])
save_fig(fig, 'fig_4_15_regtype_v1')

# v2 donut
fig, ax = plt.subplots(figsize=(5, 5))
wedges, texts, atexts = ax.pie(regs.pct, labels=regs.type, autopct='%d%%',
                                colors=['#1f6f8a', '#6d4a8f', '#8c8c8c'],
                                wedgeprops=dict(width=0.4, edgecolor='white'),
                                textprops=dict(fontsize=9))
for t in atexts:
    t.set_color('white'); t.set_fontsize(10); t.set_fontweight('bold')
ax.set_aspect('equal')
save_fig(fig, 'fig_4_15_regtype_v2_donut')